# Week 2: Vector vs raster, and map projections

**Track:** Ground Station Operator (Beginner)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/2/](https://launchdetect.com/academy/week/2/)  
**Track index:** [https://launchdetect.com/academy/ground-station-operator/](https://launchdetect.com/academy/ground-station-operator/)

---

_Vector vs raster is the fundamental data-model split in GIS. Then comes projections: how do you flatten a sphere onto a screen? This week covers Web Mercator, UTM, equirectangular, and polar stereographic, and when each is right._


## Why this week matters

Vector versus raster is not just a file-format distinction — it's the fundamental decision that shapes every operation downstream. A vector launch trajectory and a raster brightness-temperature scene answer completely different questions, and they fail in completely different ways. Picking wrong costs you minutes of compute or, worse, silently-incorrect results.

Projections compound this. Web Mercator looks fine until you measure an Antarctic ice-shelf area and get a number 6× too large. UTM looks fine until your launch trajectory crosses a zone boundary and your distances jump. Equirectangular is honest about what it is but useless for measurement. **The right projection is the one that preserves the property your current operation depends on** — angles, areas, or distances. You can only preserve two of the three, never all three.


## Learning objectives

By the end of this lab you will be able to:

- Distinguish vector from raster GIS data
- Pick an appropriate projection for a given task
- Understand why Web Mercator distorts polar regions
- Choose UTM, equirectangular, or polar stereographic correctly


## Setup — and why these dependencies

Same `pyproj` from Week 1 (more transformers) plus **`shapely`** for geometry objects (`Point`, `LineString`, `Polygon`) and **`geopandas`** for batched operations on collections of geometries. `geopandas` is essentially `pandas` where one column is a Shapely geometry — every spatial DataFrame you'll touch in this course is a GeoDataFrame.


In [ ]:
# Install required packages
!pip install -q leafmap[common] pyproj geopandas shapely


## The approach (and why)

Take a real Falcon 9 east-bound launch trajectory (a curve over the Atlantic), reproject it through three CRSes that get used in different stages of a launch operation:

1. **WGS84 (EPSG:4326)** — what the original trajectory data is in, what the rocket GPS receiver reports, what every public dataset publishes.
2. **Web Mercator (EPSG:3857)** — what you'd push to a public web map.
3. **UTM Zone 17N (EPSG:32617)** — what you'd use to measure the trajectory's downrange length accurately during the early ascent.

**Why this exact set?** They're the three projections you'll meet weekly. Other projections (Albers conic, polar stereographic) come up for specialized work; these three cover 90% of cases.


In [ ]:
# Week 2: reproject a Falcon 9 trajectory through 3 different CRSes.
import leafmap.foliumap as leafmap
from pyproj import Transformer

# Sample Falcon 9 east-bound trajectory points from Cape Canaveral
trajectory_lonlat = [
    (-80.60, 28.49), (-75.0, 28.5), (-65.0, 28.9), (-55.0, 30.0),
    (-45.0, 32.0), (-35.0, 35.0), (-25.0, 39.0), (-15.0, 44.0),
]

# Reproject to Web Mercator + UTM Zone 17N
to_merc = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
to_utm = Transformer.from_crs("EPSG:4326", "EPSG:32617", always_xy=True)
merc = [to_merc.transform(lon, lat) for lon, lat in trajectory_lonlat]
utm = [to_utm.transform(lon, lat) for lon, lat in trajectory_lonlat]

print("Web Mercator (first 3):", merc[:3])
print("UTM 17N (first 3):     ", utm[:3])

m = leafmap.Map(center=[32, -55], zoom=4)
m.add_geojson({
    "type": "Feature",
    "geometry": {"type": "LineString", "coordinates": [list(p) for p in trajectory_lonlat]},
    "properties": {"name": "Falcon 9 trajectory"}
}, style={"color": "#ff6b35", "weight": 4})
m

# TODO: compute the great-circle length of this trajectory, then compute the
# planar (Web Mercator) length and the UTM length. Compare.


## What just happened — and why it works

When you reprojected the trajectory three different ways, you got three different coordinate tuples for the same physical point. None of them is 'wrong' — they're answering different questions:

- **Web Mercator coords** tell you where on a Google Maps tile the point falls.
- **UTM coords** tell you the point's position to the meter inside a single 6° zone, with conformal angles and near-true distances.
- **WGS84 lat/lon** is the universal lingua franca for storage and exchange.

The leafmap render shows the trajectory as a continuous curve on a slippy-map tile backdrop. Internally leafmap is converting your WGS84 coordinates to Web Mercator pixels every frame — the curve you see is a sub-pixel approximation of the great-circle path.


## Common gotchas

- **Don't compute area in Web Mercator.** Near the poles, areas are inflated by more than 10×. Use an equal-area projection (Albers, Mollweide) or `pyproj.Geod` for area calculations.
- **UTM is only valid inside one zone.** If your geometry crosses a zone boundary (every 6° of longitude), either split it or reproject to a CRS that doesn't have zone discontinuities.
- **Antimeridian (180° / -180°) crossings break everything.** A geometry that spans the dateline shows up as a straight line across the entire map unless you explicitly handle it. Pacific-centric projections (`+proj=merc +lon_0=180`) help.


## Doing this in QGIS (alternative path)

In QGIS: load your trajectory CSV → set the project CRS to EPSG:4326 → run *Vector → Geometry Tools → $length on the projected variant* to see how the measured length changes depending on the CRS the layer is shown in. Toggle Project → Properties → CRS to switch between Web Mercator, UTM 17N, and equirectangular, and watch the trajectory's visual shape and length change. Same data; three completely different numbers.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/2/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
